# DermViT – CNN vs. ViT vs. Pretrained ViT vs. Pretrained CNN

**Module:** Concepts of Deep Learning  
**Paper:** *An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale*  
(Dosovitskiy et al., ICLR 2021)  
**Dataset:** HAM10000 – Human Against Machine with 10000 training images

---

## Research Question

> **Can a Vision Transformer outperform a classic CNN – and what does transfer learning  
> contribute to CNN and ViT respectively? Four models in direct comparison.**

---

## Project Structure

| Model | Architecture | Pretrained | Section |
|--------|-------------|-------------|----------|
| A: CNN | CNN (4 conv blocks) | No | 3 |
| B: ViT scratch | Vision Transformer | No | 4 |
| C: ViT timm | Vision Transformer | ImageNet-21k | 5 |
| D: ResNet50 | CNN (ResNet50) | ImageNet-1k | 6 |

**Two comparison axes:**
- **Architecture:** CNN vs. ViT (each scratch and pretrained)
- **Transfer learning:** scratch vs. pretrained (each CNN and ViT)

---

## Classes in the HAM10000 dataset

| Abbreviation | Name | Type |
|--------|------|-----|
| `mel`   | Melanoma | malignant |
| `bcc`   | Basal cell carcinoma | malignant |
| `akiec` | Actinic keratosis | potentially malignant |
| `bkl`   | Benign keratosis | benign |
| `nv`    | Melanocytic nevus | benign |
| `df`    | Dermatofibroma | benign |
| `vasc`  | Vascular lesion | benign |

## 0. Environment & Dataset

**Download the dataset:**
1. Create a Kaggle account: https://www.kaggle.com  
2. Dataset: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000  
3. Extract into `./data/ham10000/`  

**Expected folder structure:**
```
data/ham10000/
├── HAM10000_metadata.csv
├── HAM10000_images_part_1/   (*.jpg)
└── HAM10000_images_part_2/   (*.jpg)
```

In [ ]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'einops', 'matplotlib',
            'seaborn', 'pandas', 'scikit-learn', 'tqdm', 'Pillow', 'timm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed.')

In [2]:
import os, math, time, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, f1_score
)
from sklearn.model_selection import train_test_split

print('Imports OK.')


Imports OK.


## 1. Load & analyze dataset

In [ ]:
# Import project modules
import sys
sys.path.insert(0, '.')   # ensures ./config.py is found

import config

# Initialize global variables and load dataset
df = config.init()

In [ ]:
# Class distribution
counts = df['dx'].value_counts()
colors = ['#E24B4A','#378ADD','#EF9F27','#1D9E75','#7F77DD','#D85A30','#639922']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bars
axes[0].bar([config.CLASS_NAMES[c] for c in counts.index], counts.values, color=colors)
axes[0].set_title('Number of images per class')
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels([config.CLASS_NAMES[c] for c in counts.index], rotation=35, ha='right')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

# Pie chart with leader lines – no overlapping labels
wedges, texts, autotexts = axes[1].pie(
    counts.values,
    colors=colors,
    startangle=90,
    radius=0.8,          # slightly smaller for more room
    wedgeprops=dict(width=0.6),  # donut style – more room in the center
    pctdistance=1.35,    # percentage far outside
    autopct='%1.1f%%',
)

# Rotate and position each percentage by angle
for i, (wedge, autotext) in enumerate(zip(wedges, autotexts)):
    # Compute the angle of the segment's center
    angle = (wedge.theta2 + wedge.theta1) / 2
    angle_rad = math.radians(angle)

    # Position further out
    x = 1.35 * math.cos(angle_rad)
    y = 1.35 * math.sin(angle_rad)
    autotext.set_position((x, y))

    # Horizontal alignment depending on side
    autotext.set_horizontalalignment('left' if x > 0 else 'right')
    autotext.set_fontsize(8.5)
    autotext.set_color('black')
    autotext.set_rotation(45)

    # Leader line from segment to label
    axes[1].annotate(
        '',
        xy=(0.85 * math.cos(angle_rad), 0.85 * math.sin(angle_rad)),
        xytext=(1.25 * math.cos(angle_rad), 1.25 * math.sin(angle_rad)),
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.8)
    )

# Legend at the bottom
axes[1].legend(
    wedges,
    [config.CLASS_NAMES[c] for c in counts.index],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.18),
    ncol=2,
    fontsize=8,
    frameon=False
)

axes[1].set_title('Class distribution (%)', pad=20)
plt.savefig('01_datenuebersicht.png', dpi=100, bbox_inches='tight')
print('⚠ Heavily imbalanced: nv (melanocytic nevus) makes up ~67% of all images!')

In [ ]:
# Example images (2 per class)
fig, axes = plt.subplots(2, config.NUM_CLASSES, figsize=(14, 5))
for col, cls in enumerate(config.CLASSES):
    subset = df[df['dx'] == cls]
    for row in range(2):
        img = Image.open(subset.iloc[row]['path']).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'{cls}\n{config.CLASS_NAMES[cls]}', fontsize=8)
plt.suptitle('Example images per class', fontsize=12)
plt.tight_layout()
plt.savefig('02_beispielbilder.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Data preparation & expanded geometric augmentation

### Why expanded augmentation?

Instead of saving copies to disk (~36 GB), the training set is **expanded in
memory**: every image is combined with all 8 rotation/flip variant indices, so
**a single epoch contains all 8 geometric variants of every image** (8× the
original number of samples).

The dataset's flat index maps to a `(row, variant)` pair:

```
row     = index // 8
variant = index %  8
```

| Advantage | Explanation |
|---|---|
| No extra storage | 0 GB additional on disk – all transforms are in-memory |
| Full variant coverage | Every image is seen in all 8 variants **every** epoch |
| Fair comparison | All 4 models train on the exact same expanded sample set |

> **Note:** Each epoch is now 8× larger, so training is ~8× slower per epoch.
> Consider lowering `NUM_EPOCHS` to keep the total compute budget reasonable.

### `train_tf` vs. `val_tf`

- **`train_tf`**: `Resize` + `Normalize` (geometric augmentation is handled
  separately in the dataset via the variant index). `ColorJitter` is left
  **commented out**.
- **`val_tf`**: only `Resize` + `Normalize` – no augmentation, and the
  validation/test sets are **not** expanded.

In [ ]:
from dataset import get_dataloaders, get_timm_dataloaders

# Train / val / test split (70 / 15 / 15)
df_train, df_temp = train_test_split(df, test_size=0.30,
                                      stratify=df['label'],
                                      random_state=config.SEED)
df_val, df_test   = train_test_split(df_temp, test_size=0.50,
                                      stratify=df_temp['label'],
                                      random_state=config.SEED)
print(f'Train: {len(df_train):5d}  |  Val: {len(df_val):5d}  |  Test: {len(df_test):5d}')

# Class weights against imbalance
class_counts  = df_train['label'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * config.NUM_CLASSES

# DataLoaders for CNN + ViT scratch (64×64)
train_loader, val_loader, test_loader = get_dataloaders(
    df_train, df_val, df_test)

# DataLoaders for timm ViT (224×224)
timm_train_loader, timm_val_loader, timm_test_loader = get_timm_dataloaders(
    df_train, df_val, df_test)

## 3. Model A: CNN ("standard approach")

The CNN is the **classic approach** for image classification before the transformer era.  
It uses local **convolutional filters** that slide step by step across the image,  
detecting local patterns (edges, textures, shapes).

```
Image → [Conv → BN → ReLU → Pool] × 4 → GlobalAvgPool → Classifier
```

**Core CNN principle:** Each neuron sees only a small local region (*receptive field*).  
Global structures are only recognized in deep layers through hierarchy.

In [ ]:
from models import build_cnn

cnn = build_cnn()

# Quick test
dummy = torch.randn(2, 3, config.IMG_SIZE, config.IMG_SIZE).to(config.DEVICE)
print(f'CNN output shape: {cnn(dummy).shape}  ← expected (2, {config.NUM_CLASSES})')
cnn_params = sum(p.numel() for p in cnn.parameters())

In [8]:
from training import run_training

cnn_history, cnn_time = run_training(
    cnn, 'cnn', train_loader, val_loader, class_weights)



── Training: cnn ──────────────────────────────────────────
  Ep |   T-Loss |   T-Acc |   V-Loss |   V-Acc | Zeit
----------------------------------------------------------


   1 |   1.8138 | 40.90% |   1.6208 | 41.21% | 67s ✓


   2 |   1.7027 | 46.09% |   1.5444 | 51.80% | 69s ✓


   3 |   1.6551 | 45.58% |   1.5185 | 48.74% | 69s


   4 |   1.5992 | 47.59% |   1.4665 | 50.47% | 67s


...

  27 |   1.1519 | 59.79% |   1.1024 | 59.25% | 68s


  28 |   1.1620 | 59.26% |   1.0944 | 59.85% | 69s


  29 |   1.1372 | 59.39% |   1.1022 | 61.65% | 67s


  30 |   1.1099 | 60.26% |   1.0962 | 59.59% | 67s

Beste Val-Acc: 63.18%  |  Gesamtzeit: 33.8 min


/workspace/DermViT/training.py:117: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=config.DEVICE))


## 4. Model B: Vision Transformer (ViT)

The ViT is the **new approach** from the paper (Dosovitskiy et al., 2021).  
Instead of local convolutional filters it uses **self-attention**, relating each patch  
to every other patch – global from the very start.

```
Image → Patches (8×8) → Linear Projection → [CLS] + Pos.Emb.
      → 6× TransformerBlock (MSA + FFN + LayerNorm)
      → CLS token → MLP Head → Class
```

**Key difference from the CNN:**  
- CNN: local filters, hierarchy through depth  
- ViT: global attention, every token sees all others immediately

In [ ]:
from models import build_vit

vit = build_vit()

# Quick test
print(f'ViT output shape: {vit(dummy).shape}  ← expected (2, {config.NUM_CLASSES})')
vit_params = sum(p.numel() for p in vit.parameters())

In [10]:
vit_history, vit_time = run_training(
    vit, 'vit', train_loader, val_loader, class_weights)



── Training: vit ──────────────────────────────────────────
  Ep |   T-Loss |   T-Acc |   V-Loss |   V-Acc | Zeit
----------------------------------------------------------


   1 |   1.8737 | 38.60% |   1.8948 | 54.13% | 76s ✓


   2 |   1.8018 | 41.17% |   1.8185 | 32.56% | 74s


   3 |   1.8118 | 39.60% |   1.7880 | 42.48% | 75s


   4 |   1.7799 | 40.50% |   1.7140 | 47.80% | 78s


...

  26 |   1.3772 | 47.76% |   1.3800 | 51.60% | 77s


  27 |   1.3969 | 47.00% |   1.3772 | 51.80% | 78s


  28 |   1.3490 | 47.89% |   1.3660 | 52.00% | 79s


  29 |   1.3720 | 47.65% |   1.3715 | 52.33% | 76s


  30 |   1.3565 | 47.65% |   1.3708 | 52.20% | 77s

Beste Val-Acc: 54.13%  |  Gesamtzeit: 38.0 min


## 5. Model C: ViT with timm (Transfer Learning)

Model C uses a **pretrained ViT** from the `timm` library.  
Instead of starting from random weights like Model B, it begins with weights  
pretrained on **ImageNet-21k (14 million images)**.

```
ImageNet-21k pretrained → freeze weights → adapt head only → fine-tuning
```

**Key difference from Model B:**  
- Model B (from scratch): learns everything anew, needs many epochs  
- Model C (pretrained): weights already good, fine-tuning is enough – faster & better  

**Why is this interesting for the comparison?**  
It shows the effect of transfer learning: how much does pretrained knowledge contribute  
on a medical dataset like HAM10000?

In [ ]:
from models import build_timm_vit

vit_timm = build_timm_vit()

dummy_timm = torch.randn(2, 3, config.IMG_SIZE_TIMM, config.IMG_SIZE_TIMM).to(config.DEVICE)
print(f'timm ViT output shape: {vit_timm(dummy_timm).shape}')
timm_params = sum(p.numel() for p in vit_timm.parameters())

In [12]:
from training import run_training_pretrained_vit

timm_history, timm_time = run_training_pretrained_vit(
    vit_timm, 'vit_timm', timm_train_loader, timm_val_loader, class_weights)



── Phase 1: Nur Head trainieren (Epochen 1-10) ──
  Ep |   T-Loss |   T-Acc |   V-Loss |   V-Acc | Zeit
----------------------------------------------------------


   1 |   1.4001 | 58.37% |   1.0976 | 62.05% | 307s ✓


   2 |   1.0385 | 65.78% |   0.9170 | 69.84% | 313s ✓


   3 |   0.9152 | 68.25% |   0.8962 | 71.77% | 316s ✓


   4 |   0.8559 | 69.40% |   0.9355 | 72.97% | 310s ✓


...

  27 |   0.0259 | 98.72% |   0.8456 | 86.82% | 551s


  28 |   0.0202 | 98.79% |   0.8538 | 86.88% | 530s


  29 |   0.0226 | 98.79% |   0.8477 | 86.95% | 528s


  30 |   0.0191 | 98.94% |   0.8468 | 86.95% | 531s

Beste Val-Acc: 87.08%  |  Gesamtzeit: 231.3 min


/workspace/DermViT/training.py:203: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=config.DEVICE))


## 6. Model D: ResNet50 (Transfer Learning – CNN)

Model D is the **CNN counterpart to Model C**. Both are pretrained and use
the same two-phase fine-tuning – the only difference is the **architecture**:

| | Model C | Model D |
|---|---|---|
| Architecture | Vision Transformer | CNN (ResNet50) |
| Pretrained on | ImageNet-21k | ImageNet-1k |
| Receptive field | Global (self-attention) | Local (convolutional filters) |
| Input size | 224×224 | 224×224 |
| Training | 2-phase | 2-phase |

**This allows two questions to be answered directly:**
1. CNN scratch vs. ResNet50 → What does transfer learning contribute for CNNs?
2. ViT timm vs. ResNet50 → What does the transformer architecture contribute for pretrained models?

In [ ]:
from models import build_resnet50
from dataset import get_resnet_dataloaders

resnet50 = build_resnet50()
resnet50_params = sum(p.numel() for p in resnet50.parameters())

# DataLoaders for Model D (224×224, ImageNet normalization)
resnet_train_loader, resnet_val_loader, resnet_test_loader = get_resnet_dataloaders(
    df_train, df_val, df_test)

# Quick test
dummy_resnet = torch.randn(2, 3, config.IMG_SIZE_RESNET, config.IMG_SIZE_RESNET).to(config.DEVICE)
print(f'ResNet50 output shape: {resnet50(dummy_resnet).shape}  ← expected (2, {config.NUM_CLASSES})')

In [ ]:
from training import run_training_resnet

resnet_history, resnet_time = run_training_resnet(
    resnet50, 'resnet50', resnet_train_loader, resnet_val_loader, class_weights)


## 7. Direct comparison: training curves (all 4 models)

In [ ]:
ep     = range(1, config.NUM_EPOCHS + 1)
ep_224 = range(1, len(timm_history['val_acc']) + 1)  # 30 epochs (phase 1+2)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = {'CNN': '#D85A30', 'ViT scratch': '#534AB7',
          'ViT timm': '#1D9E75', 'ResNet50': '#EF9F27'}

# Loss
axes[0].plot(ep,     cnn_history['val_loss'],    label='CNN (scratch)',  color=colors['CNN'],        linewidth=2, linestyle='--')
axes[0].plot(ep,     vit_history['val_loss'],    label='ViT (scratch)',  color=colors['ViT scratch'],linewidth=2)
axes[0].plot(ep_224, timm_history['val_loss'],   label='ViT (timm)',     color=colors['ViT timm'],   linewidth=2, linestyle='-.')
axes[0].plot(ep_224, resnet_history['val_loss'], label='ResNet50 (timm)',color=colors['ResNet50'],   linewidth=2, linestyle=':')
axes[0].set_title('Validation loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(ep,     [a*100 for a in cnn_history['val_acc']],    label='CNN (scratch)',  color=colors['CNN'],        linewidth=2, linestyle='--')
axes[1].plot(ep,     [a*100 for a in vit_history['val_acc']],    label='ViT (scratch)',  color=colors['ViT scratch'],linewidth=2)
axes[1].plot(ep_224, [a*100 for a in timm_history['val_acc']],   label='ViT (timm)',     color=colors['ViT timm'],   linewidth=2, linestyle='-.')
axes[1].plot(ep_224, [a*100 for a in resnet_history['val_acc']], label='ResNet50 (timm)',color=colors['ResNet50'],   linewidth=2, linestyle=':')
axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('All 4 models – training curves', fontsize=14)
plt.tight_layout()
plt.savefig('03_training_vergleich.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Evaluation on the test set

In [ ]:
from visualization import get_predictions

cnn_preds,    true_labels, cnn_probs    = get_predictions(cnn,      test_loader)
vit_preds,    _,           vit_probs    = get_predictions(vit,      test_loader)
timm_preds,   _,           timm_probs   = get_predictions(vit_timm, timm_test_loader)
resnet_preds, _,           resnet_probs = get_predictions(resnet50, resnet_test_loader)

cnn_acc    = (cnn_preds    == true_labels).mean()
vit_acc    = (vit_preds    == true_labels).mean()
timm_acc   = (timm_preds   == true_labels).mean()
resnet_acc = (resnet_preds == true_labels).mean()
cnn_bacc    = balanced_accuracy_score(true_labels, cnn_preds)
vit_bacc    = balanced_accuracy_score(true_labels, vit_preds)
timm_bacc   = balanced_accuracy_score(true_labels, timm_preds)
resnet_bacc = balanced_accuracy_score(true_labels, resnet_preds)

print(f'{'Metric':<25} {'CNN':>10} {'ViT':>10} {'ViT timm':>10} {'ResNet50':>10}')
print('-' * 67)
print(f'{'Accuracy':<25} {cnn_acc:>10.2%} {vit_acc:>10.2%} {timm_acc:>10.2%} {resnet_acc:>10.2%}')
print(f'{'Balanced Accuracy':<25} {cnn_bacc:>10.2%} {vit_bacc:>10.2%} {timm_bacc:>10.2%} {resnet_bacc:>10.2%}')
print(f'{'Parameters':<25} {cnn_params:>10,} {vit_params:>10,} {timm_params:>10,} {resnet50_params:>10,}')
print(f'{'Training time (min)':<25} {cnn_time/60:>10.1f} {vit_time/60:>10.1f} {timm_time/60:>10.1f} {resnet_time/60:>10.1f}')

In [ ]:
# Confusion matrices – all 4 models
short_names = [config.CLASS_NAMES[config.IDX2CLASS[i]][:10] for i in range(config.NUM_CLASSES)]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
for ax, preds, title in zip(axes,
    [cnn_preds, vit_preds, timm_preds, resnet_preds],
    ['Model A: CNN (scratch)', 'Model B: ViT (scratch)',
     'Model C: ViT (timm)', 'Model D: ResNet50 (timm)']):
    cm   = confusion_matrix(true_labels, preds)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_n, annot=True, fmt='.0%', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticklabels(short_names, rotation=40, ha='right')
plt.suptitle('Confusion matrices – all 4 models', fontsize=13)
plt.tight_layout()
plt.savefig('04_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import f1_score
cnn_f1    = f1_score(true_labels, cnn_preds,    average=None)
vit_f1    = f1_score(true_labels, vit_preds,    average=None)
timm_f1   = f1_score(true_labels, timm_preds,   average=None)
resnet_f1 = f1_score(true_labels, resnet_preds, average=None)

x = np.arange(config.NUM_CLASSES)
w = 0.2
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(x - 1.5*w, cnn_f1,    w, label='CNN (scratch)',  color='#D85A30', alpha=0.85)
ax.bar(x - 0.5*w, vit_f1,    w, label='ViT (scratch)',  color='#534AB7', alpha=0.85)
ax.bar(x + 0.5*w, timm_f1,   w, label='ViT (timm)',     color='#1D9E75', alpha=0.85)
ax.bar(x + 1.5*w, resnet_f1, w, label='ResNet50 (timm)',color='#EF9F27', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([config.CLASS_NAMES[config.IDX2CLASS[i]] for i in range(config.NUM_CLASSES)],
                   rotation=30, ha='right')
ax.set_ylabel('F1 score')
ax.set_title('Per-class F1 score – all 4 models', fontsize=12)
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('05_f1_vergleich.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Interpretability

### 9a. ViT: Attention Rollout
Shows **where the ViT model looks** – accumulated across all transformer layers.

### 9b. CNN: Grad-CAM
Shows **which image regions activate the CNN** – via gradients of the last conv layer.

In [ ]:
from visualization import GradCAM, visualize_comparison
from dataset import get_transforms

# Grad-CAM on the last conv layer of the CNN
last_conv = list(cnn.features[-1].children())[-3]
gradcam   = GradCAM(cnn, last_conv)
_, val_tf = get_transforms()

print('GradCAM ready.')

In [ ]:
print('Generating interpretability visualizations...')
for cls_idx in range(config.NUM_CLASSES):
    idxs = np.where(true_labels == cls_idx)[0]
    if len(idxs) == 0: continue
    row       = df_test.iloc[idxs[0]]
    image_pil = Image.open(row['path']).convert('RGB')
    fig = visualize_comparison(
        vit, cnn, gradcam, image_pil, cls_idx, val_tf,
        config.CLASS_NAMES, config.IDX2CLASS,
        config.PATCH_SIZE, config.IMG_SIZE
    )
    fig.savefig(f'06_interp_{config.IDX2CLASS[cls_idx]}.png',
                dpi=100, bbox_inches='tight')
    plt.show()

## 10. Final summary

In [ ]:
print('=' * 78)
print('  COMPARISON: CNN vs. ViT (scratch) vs. ViT (timm) vs. ResNet50 (timm)')
print('=' * 78)
rows = [
    ('Test accuracy',      f'{cnn_acc:.2%}',       f'{vit_acc:.2%}',       f'{timm_acc:.2%}',      f'{resnet_acc:.2%}'),
    ('Balanced Accuracy',  f'{cnn_bacc:.2%}',      f'{vit_bacc:.2%}',      f'{timm_bacc:.2%}',     f'{resnet_bacc:.2%}'),
    ('Parameters',         f'{cnn_params:,}',       f'{vit_params:,}',      f'{timm_params:,}',     f'{resnet50_params:,}'),
    ('Training time',      f'{cnn_time/60:.1f}m',   f'{vit_time/60:.1f}m',  f'{timm_time/60:.1f}m', f'{resnet_time/60:.1f}m'),
    ('Architecture',       'CNN',                   'Transformer',          'Transformer',          'CNN'),
    ('Pretrained',         'No',                    'No',                   'ImageNet-21k',         'ImageNet-1k'),
    ('Receptive field',    'Local',                 'Global',               'Global',               'Local'),
]
header = f'  {"Metric":<22} {"CNN":>12} {"ViT":>12} {"ViT timm":>12} {"ResNet50":>12}'
print(header)
print('  ' + '-' * 72)
for r in rows:
    print(f'  {r[0]:<22} {r[1]:>12} {r[2]:>12} {r[3]:>12} {r[4]:>12}')
print('=' * 78)

all_accs  = [cnn_acc, vit_acc, timm_acc, resnet_acc]
all_names = ['CNN (scratch)', 'ViT (scratch)', 'ViT (timm)', 'ResNet50 (timm)']
best      = all_names[all_accs.index(max(all_accs))]
print(f'\n→ Best test accuracy: {best} ({max(all_accs):.2%})')
print(f'\nKey questions:')
print(f'  Transfer learning (CNN):  {resnet_acc:.2%} vs {cnn_acc:.2%} (+{(resnet_acc-cnn_acc)*100:.1f}%)')
print(f'  Transfer learning (ViT):  {timm_acc:.2%} vs {vit_acc:.2%} (+{(timm_acc-vit_acc)*100:.1f}%)')
print(f'  Architecture (pretrained): {timm_acc:.2%} vs {resnet_acc:.2%} ({(timm_acc-resnet_acc)*100:+.1f}%)')
print(f'  Architecture (scratch):    {vit_acc:.2%} vs {cnn_acc:.2%} ({(vit_acc-cnn_acc)*100:+.1f}%)')

In [ ]:
# Final comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models    = ['CNN\n(scratch)', 'ViT\n(scratch)', 'ViT\n(timm)', 'ResNet50\n(timm)']
colors_b  = ['#D85A30', '#534AB7', '#1D9E75', '#EF9F27']

for ax, vals, title in zip(axes,
    [[cnn_acc*100, vit_acc*100, timm_acc*100, resnet_acc*100],
     [cnn_bacc*100, vit_bacc*100, timm_bacc*100, resnet_bacc*100],
     [cnn_time/60, vit_time/60, timm_time/60, resnet_time/60]],
    ['Test accuracy (%)', 'Balanced Accuracy (%)', 'Training time (min)']):
    bars = ax.bar(models, vals, color=colors_b, width=0.5)
    ax.set_title(title, fontsize=11)
    if 'Accuracy' in title or 'accuracy' in title:
        ax.set_ylim(0, 100)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                f'{b.get_height():.1f}', ha='center', fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('DermViT – all 4 models compared', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('07_zusammenfassung.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Discussion & Conclusion

Extend this cell with your own observations after your experiments.

1. **CNN vs. ViT (scratch):** Did the custom-implemented ViT beat the CNN? Why / why not?
2. **Transfer learning:** How much better is ViT (timm) compared to ViT (scratch)? What does this say about the value of pretrained weights?
3. **Balanced accuracy:** Does the result differ from regular accuracy? What does this say about class imbalance?
4. **Attention:** Do ViT scratch and ViT timm look at the same image regions? Compare the attention maps.
5. **Difficult classes:** Which classes do all three models confuse most often?
6. **Trade-off:** ViT timm has the most parameters and needs 224px – is the extra effort justified?
7. **Phase 1 vs. Phase 2:** Did fine-tuning (Phase 2) significantly improve accuracy for Model C?
8. **Clinical relevance:** In melanoma detection, a false negative (sick classified as healthy) is worse than a false positive – which model is best there?